# Colab launcher

Set **Runtime > Change runtime type > GPU** before running. Then run the single cell below.

This verbose version prints each stage live: GPU check, CUDA check, clone, file listing, build, and program launch.


In [ ]:
#@title Build and run the CUDA seed tool
REPO_URL = "https://github.com/cZarVoid/comission-colab.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
LARGE_BIOMES = False #@param {type:"boolean"}
UNBOUND = False #@param {type:"boolean"}
PRINT_INTERVAL = 256 #@param {type:"integer"}
CUDA_ARCH = "auto" #@param ["auto", "sm_75", "sm_80", "sm_86", "sm_89", "sm_90"]
RUN_ARGS = "--device 0" #@param {type:"string"}
DISCORD_WEBHOOK_URL = "" #@param {type:"string"}
DISCORD_MESSAGE_PREFIX = "Seed output" #@param {type:"string"}
SEND_DISCORD_STATUS = True #@param {type:"boolean"}

import json
import os
import shlex
import subprocess
import sys
import time
import urllib.request


def print_flush(message=""):
    print(message, flush=True)


def discord_status(message):
    """Send a small status message to Discord without exposing the webhook in logs."""
    if not SEND_DISCORD_STATUS or not webhook:
        return
    payload = json.dumps({"content": message}).encode("utf-8")
    req = urllib.request.Request(
        webhook,
        data=payload,
        headers={"Content-Type": "application/json", "User-Agent": "colab-cuda-launcher"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=10) as response:
            response.read()
    except Exception as exc:
        print_flush(f"[warning] Discord status message failed: {type(exc).__name__}: {exc}")


env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["LARGE_BIOMES"] = "1" if LARGE_BIOMES else "0"
env["UNBOUND"] = "1" if UNBOUND else "0"
env["PRINT_INTERVAL"] = str(PRINT_INTERVAL)

if CUDA_ARCH != "auto":
    env["CUDA_ARCH"] = CUDA_ARCH

# Prefer a pasted value in the form field. If left blank, try a Colab Secret
# named DISCORD_WEBHOOK_URL. Do not commit webhook URLs into public GitHub repos.
webhook = DISCORD_WEBHOOK_URL.strip()
if not webhook:
    try:
        from google.colab import userdata
        webhook = (userdata.get("DISCORD_WEBHOOK_URL") or "").strip()
    except Exception:
        webhook = ""

if webhook:
    env["DISCORD_WEBHOOK_URL"] = webhook
    env["DISCORD_MESSAGE_PREFIX"] = DISCORD_MESSAGE_PREFIX

print_flush("=== Colab CUDA launcher starting ===")
print_flush(f"Repo: {REPO_URL}")
print_flush(f"Branch: {BRANCH}")
print_flush(f"LARGE_BIOMES={env['LARGE_BIOMES']} UNBOUND={env['UNBOUND']} PRINT_INTERVAL={env['PRINT_INTERVAL']}")
print_flush(f"CUDA_ARCH={env.get('CUDA_ARCH', 'auto')}")
print_flush(f"RUN_ARGS={RUN_ARGS}")
print_flush("Discord webhook: configured" if webhook else "Discord webhook: not configured")

if webhook:
    discord_status("Colab launcher started.")

cmd = f"""
set -euo pipefail

log_step() {{
  echo
  echo "=== $1 ==="
}}

log_step "GPU info"
nvidia-smi || true

log_step "CUDA compiler"
which nvcc || true
nvcc --version || true

log_step "Cloning repo"
rm -rf /content/comission_cuda
git clone --depth 1 --branch {shlex.quote(BRANCH)} {shlex.quote(REPO_URL)} /content/comission_cuda

log_step "Repo files"
ls -la /content/comission_cuda

log_step "cuda_com files"
ls -la /content/comission_cuda/cuda_com

log_step "Entering cuda_com"
cd /content/comission_cuda/cuda_com
chmod +x ./colab_run.sh ./discord_output_bridge.py 2>/dev/null || true

log_step "Running launcher"
echo "Command: ./colab_run.sh {RUN_ARGS}"
./colab_run.sh {RUN_ARGS}
"""

print_flush("\n=== Shell command prepared; starting live output ===")
start = time.time()
process = subprocess.Popen(
    ["bash", "-lc", cmd],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
finally:
    return_code = process.wait()

elapsed = time.time() - start
print_flush(f"\n=== Launcher finished with exit code {return_code} after {elapsed:.1f}s ===")

if return_code == 0:
    discord_status("Colab launcher finished normally.")
else:
    discord_status(f"Colab launcher exited with code {return_code}.")
    raise RuntimeError(f"Launcher exited with code {return_code}")
